# Oneway (pre-model) analysis

This notebook builds a BayesOptModel on a train/test split and generates
pre-model oneway plots (no predictions). Update `cfg_path` to point to
the config you want to analyze.


In [ ]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if not (repo_root / 'ins_pricing').exists():
    for parent in repo_root.parents:
        if (parent / 'ins_pricing').exists():
            repo_root = parent
            sys.path.insert(0, str(repo_root))
            break

os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'


In [ ]:
import json
import pandas as pd

from ins_pricing import bayesopt as ropt
from ins_pricing.cli.utils.cli_common import split_train_test


In [ ]:
cfg_path = repo_root / 'ins_pricing/examples/modelling/config_ft_unsupervised_resn.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8'))

model_name = f"{cfg['model_list'][0]}_{cfg['model_categories'][0]}"
data_dir = (cfg_path.parent / cfg['data_dir']).resolve()
raw_path = data_dir / f"{model_name}.csv"
raw = pd.read_csv(raw_path)
raw.fillna(0, inplace=True)

holdout_ratio = cfg.get('holdout_ratio', cfg.get('prop_test', 0.25))
split_strategy = cfg.get('split_strategy', 'random')
split_group_col = cfg.get('split_group_col')
split_time_col = cfg.get('split_time_col')
split_time_ascending = cfg.get('split_time_ascending', True)
rand_seed = cfg.get('rand_seed', 13)

train_df, test_df = split_train_test(
    raw,
    holdout_ratio=holdout_ratio,
    strategy=split_strategy,
    group_col=split_group_col,
    time_col=split_time_col,
    time_ascending=split_time_ascending,
    rand_seed=rand_seed,
    reset_index_mode='time_group',
    ratio_label='holdout_ratio',
)

output_dir = cfg.get('output_dir', './ResultsOnewayPre')
output_dir = str((cfg_path.parent / output_dir).resolve())

model = ropt.BayesOptModel(
    train_df,
    test_df,
    model_name,
    cfg['target'],
    cfg['weight'],
    cfg['feature_list'],
    task_type=cfg.get('task_type', 'regression'),
    binary_resp_nme=cfg.get('binary_resp_nme'),
    cate_list=cfg.get('categorical_features', []),
    prop_test=cfg.get('prop_test', 0.25),
    rand_seed=cfg.get('rand_seed', 13),
    epochs=cfg.get('epochs', 50),
    use_gpu=cfg.get('use_gpu', True),
    use_resn_data_parallel=False,
    use_ft_data_parallel=False,
    use_gnn_data_parallel=False,
    use_resn_ddp=False,
    use_ft_ddp=False,
    use_gnn_ddp=False,
    output_dir=output_dir,
    infer_categorical_max_unique=cfg.get('infer_categorical_max_unique', 50),
    infer_categorical_max_ratio=cfg.get('infer_categorical_max_ratio', 0.05),
)

n_bins = cfg.get('plot', {}).get('n_bins', 10)
model.plot_oneway(n_bins=n_bins, plot_subdir='oneway/pre')
print(f'Plots saved under: {output_dir}/plot/{model_name}/oneway/pre')
